In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Google Drive mounted!")

Mounted at /content/drive
Google Drive mounted!


In [2]:
import pandas as pd
import numpy as np

INPUT_FILE  = "/content/drive/MyDrive/insider_threat/data/master_log.csv"
OUTPUT_FILE = "/content/drive/MyDrive/insider_threat/data/user_features.csv"

print("Libraries loaded!")
print("Setup complete!")

Libraries loaded!
Setup complete!


In [3]:
print("Loading master log...")

log = pd.read_csv(INPUT_FILE)
log["datetime"]  = pd.to_datetime(log["datetime"])
log["date_only"] = pd.to_datetime(log["date_only"]).dt.date

logon_log  = log[log["source"] == "logon"].copy()
file_log   = log[log["source"] == "file"].copy()
email_log  = log[log["source"] == "email"].copy()
device_log = log[log["source"] == "device"].copy()

users = sorted(log["user"].unique())

print(f"Total events : {len(log):,}")
print(f"Total users  : {log['user'].nunique():,}")
print(f"Logon events : {len(logon_log):,}")
print(f"File events  : {len(file_log):,}")
print(f"Email events : {len(email_log):,}")
print(f"Device events: {len(device_log):,}")
print("Done!")

Loading master log...
Total events : 541,985
Total users  : 500
Logon events : 65,400
File events  : 196,304
Email events : 278,521
Device events: 1,760
Done!


In [4]:
print("Computing login features...")

login_features = (
    logon_log.groupby("user")
    .agg(
        total_logins       = ("id",           "count"),
        unique_days_active = ("date_only",    "nunique"),
        off_hours_logins   = ("is_off_hours", "sum"),
        weekend_logins     = ("is_weekend",   "sum"),
        unique_pcs_used    = ("pc",           "nunique"),
        avg_login_hour     = ("hour",         "mean"),
    )
    .reset_index()
)

login_features["off_hours_login_rate"] = (
    login_features["off_hours_logins"] / login_features["total_logins"].clip(1)
)
login_features["weekend_login_rate"] = (
    login_features["weekend_logins"] / login_features["total_logins"].clip(1)
)

print(f"Login features: {login_features.shape[1]-1} columns")
print("Done!")

Computing login features...
Login features: 8 columns
Done!


In [5]:
print("Computing file access features...")

sensitive_ext = ["zip", "csv", "pdf"]
file_log["is_sensitive"] = file_log["filename"].str.split(".").str[-1].isin(sensitive_ext)
file_log["is_delete"]    = file_log["activity"] == "delete"
file_log["is_copy"]      = file_log["activity"] == "copy"

file_features = (
    file_log.groupby("user")
    .agg(
        total_file_events     = ("id",           "count"),
        sensitive_file_access = ("is_sensitive", "sum"),
        file_deletions        = ("is_delete",    "sum"),
        file_copies           = ("is_copy",      "sum"),
        off_hours_file_access = ("is_off_hours", "sum"),
    )
    .reset_index()
)

file_features["sensitive_file_rate"] = (
    file_features["sensitive_file_access"] / file_features["total_file_events"].clip(1)
)
file_features["off_hours_file_rate"] = (
    file_features["off_hours_file_access"] / file_features["total_file_events"].clip(1)
)

print(f"File features: {file_features.shape[1]-1} columns")
print("Done!")

Computing file access features...
File features: 7 columns
Done!


In [6]:
print("Computing email features...")

email_log["is_external"] = ~email_log["to"].str.contains("company.com", na=False)
email_log["has_attach"]  = email_log["attachments"].fillna(0) > 0
email_log["size"]        = email_log["size"].fillna(0)

email_features = (
    email_log.groupby("user")
    .agg(
        total_emails            = ("id",          "count"),
        external_emails         = ("is_external", "sum"),
        emails_with_attachments = ("has_attach",  "sum"),
        avg_email_size          = ("size",         "mean"),
        off_hours_emails        = ("is_off_hours", "sum"),
    )
    .reset_index()
)

email_features["external_email_rate"] = (
    email_features["external_emails"] / email_features["total_emails"].clip(1)
)
email_features["attachment_rate"] = (
    email_features["emails_with_attachments"] / email_features["total_emails"].clip(1)
)
email_features["off_hours_email_rate"] = (
    email_features["off_hours_emails"] / email_features["total_emails"].clip(1)
)

print(f"Email features: {email_features.shape[1]-1} columns")
print("Done!")

Computing email features...
Email features: 8 columns
Done!


In [7]:
print("Computing device/USB features...")

device_features = (
    device_log.groupby("user")
    .agg(
        usb_connections = ("id",           "count"),
        off_hours_usb   = ("is_off_hours", "sum"),
    )
    .reset_index()
)

device_features["off_hours_usb_rate"] = (
    device_features["off_hours_usb"] / device_features["usb_connections"].clip(1)
)

print(f"Device features: {device_features.shape[1]-1} columns")
print("Done!")

Computing device/USB features...
Device features: 3 columns
Done!


In [8]:
print("Merging all feature groups...")

base = pd.DataFrame({"user": users})

features = (
    base
    .merge(login_features,  on="user", how="left")
    .merge(file_features,   on="user", how="left")
    .merge(email_features,  on="user", how="left")
    .merge(device_features, on="user", how="left")
)

features = features.fillna(0)

print(f"Final feature table: {features.shape[0]} users x {features.shape[1]-1} features")
print("\nFeature columns:")
for i, col in enumerate([c for c in features.columns if c != "user"], 1):
    print(f"  {i:2d}. {col}")

Merging all feature groups...
Final feature table: 500 users x 26 features

Feature columns:
   1. total_logins
   2. unique_days_active
   3. off_hours_logins
   4. weekend_logins
   5. unique_pcs_used
   6. avg_login_hour
   7. off_hours_login_rate
   8. weekend_login_rate
   9. total_file_events
  10. sensitive_file_access
  11. file_deletions
  12. file_copies
  13. off_hours_file_access
  14. sensitive_file_rate
  15. off_hours_file_rate
  16. total_emails
  17. external_emails
  18. emails_with_attachments
  19. avg_email_size
  20. off_hours_emails
  21. external_email_rate
  22. attachment_rate
  23. off_hours_email_rate
  24. usb_connections
  25. off_hours_usb
  26. off_hours_usb_rate


In [9]:
features.to_csv(OUTPUT_FILE, index=False)

print("=" * 50)
print("  FEATURE ENGINEERING COMPLETE")
print("=" * 50)
print(f"  Users          : {features.shape[0]}")
print(f"  Features       : {features.shape[1]-1}")
print(f"  Output file    : user_features.csv")
print("=" * 50)
print("Saved to Google Drive!")
print("Next step: Run 04_model notebook!")
features.head()

  FEATURE ENGINEERING COMPLETE
  Users          : 500
  Features       : 26
  Output file    : user_features.csv
Saved to Google Drive!
Next step: Run 04_model notebook!


,user,total_logins,unique_days_active,off_hours_logins,weekend_logins,unique_pcs_used,avg_login_hour,off_hours_login_rate,weekend_login_rate,total_file_events,...,external_emails,emails_with_attachments,avg_email_size,off_hours_emails,external_email_rate,attachment_rate,off_hours_email_rate,usb_connections,off_hours_usb,off_hours_usb_rate
0,U0001,124,63,2,2,101,12.483871,0.016129,0.016129,328,...,0,112,566819.082585,191,0.0,0.201077,0.342908,0.0,0.0,0.0
1,U0002,129,62,1,1,94,12.100775,0.007752,0.007752,326,...,0,105,544845.466797,121,0.0,0.205078,0.236328,5.0,1.0,0.2
2,U0003,144,65,9,9,101,11.965278,0.062500,0.062500,352,...,0,109,515985.167577,165,0.0,0.198543,0.300546,5.0,3.0,0.6
3,U0004,123,63,1,1,97,11.894309,0.008130,0.008130,326,...,0,106,571380.468750,133,0.0,0.194853,0.244485,4.0,0.0,0.0
4,U0005,133,65,3,3,95,12.789474,0.022556,0.022556,352,...,0,93,460623.948617,117,0.0,0.183794,0.231225,2.0,1.0,0.5
